In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import os

# --- 1. Helper Functions ---

def sanitize_data(df):
    """Fills NaNs and clips extreme outliers to stabilize the manifold."""
    df = df.copy()
    df = df.fillna(0)
    cols = [f"f{i}" for i in range(26)]
    # Clipping at -10, 10 to keep gradients from exploding while preserving signal
    df[cols] = np.clip(df[cols], -10, 10)
    return df

def build_normalized_kinematics(df, features):
    """Calculates Position, Velocity, and Acceleration with Cross-Symbol Normalization."""
    print("Extracting Normalized Kinematics (v and a)...")
    df = df.copy()
    
    for f in features:
        # Velocity: Difference between time steps
        v = df.groupby('symbol_id')[f].diff().fillna(0)
        # Normalize by symbol-specific standard deviation to handle varying volatility
        v_std = v.groupby(df['symbol_id']).transform('std').replace(0, 1)
        df[f'{f}_vel'] = (v / v_std).astype(np.float32)
        
        # Acceleration: Change in velocity
        a = df.groupby('symbol_id')[f'{f}_vel'].diff().fillna(0)
        a_std = a.groupby(df['symbol_id']).transform('std').replace(0, 1)
        df[f'{f}_acc'] = (a / a_std).astype(np.float32)
        
        # Path: 5-step rolling mean to capture short-term 'memory'
        df[f'{f}_path'] = df.groupby('symbol_id')[f].transform(lambda x: x.rolling(5).mean()).fillna(0).astype(np.float32)
        
    return df

def get_entropy_weights(df, features):
    """Assigns higher importance to high-energy (surprising) feature movements."""
    print("Calculating Information Theory Weights...")
    vel_cols = [f'{f}_vel' for f in features]
    # Total Kinetic Energy of the vector
    kinetic_energy = np.sqrt((df[vel_cols]**2).sum(axis=1))
    
    # log1p prevents extreme energy from drowning out everything else
    weights = np.log1p(kinetic_energy)
    return (weights / weights.mean()).astype(np.float32)

# --- 2. Main Audit & Execution Pipeline ---

def run_final_audit(train_df, test_df):
    FEATURE_COLS = [f"f{i}" for i in range(26)]
    
    # Step 1: Feature Expansion
    train_k = build_normalized_kinematics(sanitize_data(train_df), FEATURE_COLS)
    test_k = build_normalized_kinematics(sanitize_data(test_df), FEATURE_COLS)
    
    EXTENDED_FEATURES = FEATURE_COLS + \
                        [f'{f}_vel' for f in FEATURE_COLS] + \
                        [f'{f}_acc' for f in FEATURE_COLS] + \
                        [f'{f}_path' for f in FEATURE_COLS]
    
    # Step 2: Information Entropy Filter
    weights = get_entropy_weights(train_k, FEATURE_COLS)
    
    # Step 3: 80/20 Temporal Split
    split_idx = int(len(train_k) * 0.8)
    
    X_train = train_k.iloc[:split_idx][EXTENDED_FEATURES]
    y_train = train_k.iloc[:split_idx]['y']
    w_train = weights.iloc[:split_idx]
    
    X_val = train_k.iloc[split_idx:][EXTENDED_FEATURES]
    y_val = train_k.iloc[split_idx:]['y']
    
    # Step 4: Model Configuration
    # Huber loss is used because it's robust to high-kurtosis 'Black Swan' outliers
    params = {
        'objective': 'huber',
        'alpha': 0.1,
        'metric': 'rmse',
        'verbosity': -1,
        'learning_rate': 0.03,
        'num_leaves': 63,
        'feature_fraction': 0.8,
        'n_jobs': 4  # Match your 4-core CPU
    }
    
    print(f"Training on {len(X_train)} samples... validating on {len(X_val)}")
    dtrain = lgb.Dataset(X_train, label=y_train, weight=w_train)
    dval = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(
        params, 
        dtrain, 
        num_boost_round=1500, 
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(stopping_rounds=50)]
    )
    
    # Step 5: Validation Check
    val_preds = model.predict(X_val) * 0.5 # 0.5 Shrinkage for safety
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    
    print("\n" + "="*40)
    print(f"FINAL AUDIT RMSE: {val_rmse:.6f}")
    print("="*40 + "\n")
    
    # Step 6: Test Inference & Submission
    print("Generating submission_kinematic_final.csv...")
    test_preds = model.predict(test_k[EXTENDED_FEATURES]) * 0.5
    submission = pd.DataFrame({'Id': test_df['Id'], 'y': test_preds})
    submission.to_csv("submission_kinematic_final.csv", index=False)
    
    return val_rmse

# RUN THE WHOLE THING
# run_final_audit(train_df, test_df)

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [7]:
run_final_audit(train_df, test_df)

Extracting Normalized Kinematics (v and a)...
Extracting Normalized Kinematics (v and a)...
Calculating Information Theory Weights...
Training on 1613701 samples... validating on 403426
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[73]	valid_0's rmse: 0.00224512

FINAL AUDIT RMSE: 0.002247

Generating submission_kinematic_final.csv...


KeyError: 'id'